<a href="https://colab.research.google.com/github/swatijaincts/GENAI-Course/blob/main/GL_graphrag_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Great Learning GraphRAG Lab



## 1. Setup


In [ ]:
!pip install matplotlib pandas networkx pyvis google-genai sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.5 MB/s eta 0:00:00


In [ ]:
import html
import json
import math
import re
import os
import urllib.request

import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import HTML, display
from pyvis.network import Network
import pandas as pd
from networkx.algorithms import community as nx_comm

print("Imports OK")


Imports OK


## 2. Connecting the LLMs

| `PROVIDER` / `EMBEDDER` | Chat | Embeddings |
|-------------------------|------|------------|
| `"ollama"` | `phi4-mini` | `nomic-embed-text` |
| `"gemini"` | `gemini-2.0-flash` | `gemini-embedding-001` |
| `"minilm"` (embed only) | — | local MiniLM |

Gemini needs an API key. Ollama must be running with the models pulled.

Choose provider + embedder. Paste Gemini key if needed.

In [ ]:
PROVIDER = "ollama"  # "ollama" or "gemini"
EMBEDDER = PROVIDER  # or "minilm"

OLLAMA_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "phi4-mini"
OLLAMA_EMBED_MODEL = "nomic-embed-text"

GEMINI_MODEL = "gemini-2.0-flash"
GEMINI_EMBED_MODEL = "gemini-embedding-001"
GEMINI_API_KEY = ""  # required if PROVIDER/EMBEDDER is gemini

MINILM_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("PROVIDER =", PROVIDER, "| EMBEDDER =", EMBEDDER)

PROVIDER = ollama | EMBEDDER = ollama


Chat via Ollama HTTP API.

In [ ]:
def ask_ollama(prompt: str, system: str, *, json_mode: bool = False) -> str:
    payload = {
        "model": OLLAMA_MODEL,
        "stream": False,
        "options": {"temperature": 0.1 if json_mode else 0.2, "num_predict": 220},
        "messages": [
            {"role": "system", "content": system},
            {"role": "user", "content": prompt},
        ],
    }
    if json_mode:
        payload["format"] = "json"
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/chat",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        data = json.load(r)
    text = (data.get("message") or {}).get("content", "").strip()
    if not text:
        raise RuntimeError("Ollama returned an empty reply")
    return text


Chat via Gemini API.

In [ ]:
def ask_gemini(prompt: str, system: str) -> str:
    from google import genai
    from google.genai import types

    key = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY", "")
    if not key:
        raise ValueError("Set GEMINI_API_KEY in the config cell (or as an env var).")

    client = genai.Client(api_key=key)
    resp = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=0.2,
            max_output_tokens=220,
        ),
    )
    text = (resp.text or "").strip()
    if not text:
        raise RuntimeError("Gemini returned an empty reply")
    return text

In [ ]:
def ask_llm(
    prompt: str,
    system: str = "Answer briefly using only the provided context.",
    *,
    json_mode: bool = False,
) -> str:
    if PROVIDER == "ollama":
        return ask_ollama(prompt, system, json_mode=json_mode)
    if PROVIDER == "gemini":
        # Gemini: schema reminder is in the system prompt; no separate flag needed
        return ask_gemini(prompt, system)
    raise ValueError(f"Unknown PROVIDER={PROVIDER!r}. Use 'ollama' or 'gemini'.")


Probe chat — must succeed before continuing.

In [ ]:
probe = ask_llm("Reply with exactly: OK", system="Reply with one word only.")
print("LLM ready |", PROVIDER, "|", probe)

URLError: <urlopen error [Errno 111] Connection refused>

### Embedder

Real vectors for Vector RAG (not bag-of-words).

Ollama embeddings (`nomic-embed-text`).

In [ ]:
def embed_ollama(texts: list[str]) -> list[list[float]]:
    payload = {"model": OLLAMA_EMBED_MODEL, "input": texts}
    req = urllib.request.Request(
        f"{OLLAMA_URL}/api/embed",
        data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=120) as r:
        return json.load(r)["embeddings"]

Gemini embeddings.

In [ ]:
def embed_gemini(texts: list[str]) -> list[list[float]]:
    from google import genai
    from google.genai import types

    key = GEMINI_API_KEY or os.environ.get("GEMINI_API_KEY", "")
    if not key:
        raise ValueError("Set GEMINI_API_KEY for Gemini embeddings.")
    client = genai.Client(api_key=key)
    result = client.models.embed_content(
        model=GEMINI_EMBED_MODEL,
        contents=texts,
        config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
    )
    return [e.values for e in result.embeddings]

Local MiniLM (optional offline embedder).

In [ ]:
_minilm = None


def embed_minilm(texts: list[str]) -> list[list[float]]:
    global _minilm
    from sentence_transformers import SentenceTransformer

    if _minilm is None:
        _minilm = SentenceTransformer(MINILM_MODEL)
    return _minilm.encode(texts, normalize_embeddings=True).tolist()

Dispatch + smoke test.

In [ ]:
def embed(texts: list[str]) -> list[list[float]]:
    if EMBEDDER == "ollama":
        return embed_ollama(texts)
    if EMBEDDER == "gemini":
        return embed_gemini(texts)
    if EMBEDDER == "minilm":
        return embed_minilm(texts)
    raise ValueError(f"Unknown EMBEDDER={EMBEDDER!r}")


_v = embed(["hello embeddings"])
print(f"EMBEDDER={EMBEDDER} → dim={len(_v[0])}")

URLError: <urlopen error [Errno 111] Connection refused>

## 3. Documents

Toy archive: **Sales · Factory · Maintenance · Procurement**.

Executive question.

In [ ]:
QUERY = "Why was the custom BMW chassis order delayed by three weeks in August?"
print(QUERY)

Four silo chunks — edit live if you like.

In [ ]:
CHUNKS = [
    {
        "id": "A",
        "silo": "Sales CRM",
        "title": "Order #402 escalation",
        "text": "Order #402 was delayed three weeks. The client asked Sales for a penalty waiver.",
    },
    {
        "id": "B",
        "silo": "Factory Floor",
        "title": "Press Cell 4 schedule",
        "text": "Order #402 was scheduled on Press Cell 4. Press Cell 4 logged no fault that day.",
    },
    {
        "id": "C",
        "silo": "Maintenance",
        "title": "Press Cell 4 halt",
        "text": "Press Cell 4 halted because the hydraulic pump seal failed. Press Cell 4 needs a replacement seal.",
    },
    {
        "id": "D",
        "silo": "Procurement",
        "title": "Seals at port",
        "text": "The replacement seal for Press Cell 4 is stuck at the port under a customs hold.",
    },
]


Show the corpus.

In [ ]:
pd.DataFrame(CHUNKS)[["id", "silo", "title", "text"]]

## 4. Vector RAG



Embed the query.

In [ ]:
q_vec = embed([QUERY])[0]
print("dim =", len(q_vec))
print("QUERY first 8:", [round(x, 4) for x in q_vec[:8]], "...")

Embed each chunk.

In [ ]:
docs = [ch["title"] + ". " + ch["text"] for ch in CHUNKS]
chunk_vecs = embed(docs)

for ch, v in zip(CHUNKS, chunk_vecs):
    print(f"{ch['id']} first 8:", [round(x, 4) for x in v[:8]], "...")

Preview table of embedding snippets.

In [ ]:
pd.DataFrame(
    [{"id": "QUERY", "preview": str([round(x, 3) for x in q_vec[:6]]) + " …"}]
    + [
        {"id": ch["id"], "preview": str([round(x, 3) for x in v[:6]]) + " …"}
        for ch, v in zip(CHUNKS, chunk_vecs)
    ]
)

### Cosine similarity

$$
\cos(\mathbf{q}, \mathbf{d})
= \frac{\mathbf{q} \cdot \mathbf{d}}{\|\mathbf{q}\|\,\|\mathbf{d}\|}
= \frac{\sum_i q_i d_i}{\sqrt{\sum_i q_i^2}\,\sqrt{\sum_i d_i^2}}
$$

- **1** → same direction · **0** → unrelated · **-1** → opposite

Define cosine.

In [ ]:
def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(y * y for y in b))
    return dot / (na * nb) if na and nb else 0.0

Score query ↔ each chunk.

In [ ]:
ranked = []
for ch, dv in zip(CHUNKS, chunk_vecs):
    ranked.append({**ch, "score": round(cosine(q_vec, dv), 4)})
ranked.sort(key=lambda r: (-r["score"], r["id"]))

Ranked table.

In [ ]:
pd.DataFrame(
    [{"chunk": r["id"], "silo": r["silo"], "cosine": r["score"], "title": r["title"]} for r in ranked]
)

### Thresholds matter

A mid cosine (~0.5) is not proof of relevance.

Distractor texts (not the delay story).

In [ ]:
DISTRACTORS = [
    "This is a great factory with excellent parking and a popular cafeteria.",
    "BMW lifestyle ads ran in Munich last summer; dealerships reported strong foot traffic.",
    "The quarterly wellness survey asked staff about shift length and cafeteria menus.",
    "Supply chain podcasts discuss ports, seals, and customs in generic terms only.",
    "Our CRM reminds account managers to send birthday cards to key contacts.",
]

Score distractors; pick the one closest to 0.5.

In [ ]:
d_vecs = embed(DISTRACTORS)
distractor_rows = []
for text, v in zip(DISTRACTORS, d_vecs):
    score = round(cosine(q_vec, v), 4)
    distractor_rows.append({"cosine": score, "abs_from_0.5": round(abs(score - 0.5), 4), "text": text})
distractor_rows.sort(key=lambda r: r["abs_from_0.5"])
best = distractor_rows[0]
print("Closest to ~0.5:", best["cosine"])
print(best["text"])

Compare to threshold vs real top chunk.

In [ ]:
THRESHOLD = 0.5
print("Top chunk", ranked[0]["id"], "cosine =", ranked[0]["score"])
print("Distractor cosine =", best["cosine"], "| threshold =", THRESHOLD)
print("Distractor passes threshold?", best["cosine"] >= THRESHOLD)
pd.DataFrame(distractor_rows)[["cosine", "text"]]

### Generate (classic Vector RAG)

Top-2 chunks → LLM.

Select top-2 context.

In [ ]:
TOP_K = 2
top_k = ranked[:TOP_K]
for r in top_k:
    print(f"[{r['id']}] cosine={r['score']}  {r['silo']}: {r['text']}")

Build the RAG prompt.

In [ ]:
context = "\n\n".join(
    f"[{r['id']} | {r['silo']}] {r['title']}: {r['text']}" for r in top_k
)
rag_prompt = (
    f"Question: {QUERY}\n\n"
    f"Retrieved documents:\n{context}\n\n"
    "Answer using only the retrieved documents. If the root cause is not in them, say so."
)
print(rag_prompt)

Call the LLM.

In [ ]:
vector_rag_answer = ask_llm(
    rag_prompt,
    system="You are a RAG assistant. Use only the retrieved documents. Be concise (3 sentences max).",
)
print(f"--- Vector RAG ({PROVIDER}, top-{TOP_K}) ---")
print(vector_rag_answer)

## 5. Knowledge graph Extraction

LLM reads **each document** and extracts entities + relations.
We merge all these subgraphs.

Silo colors for plotting.

In [ ]:
SILO_COLORS = {
    "sales": "#4C8BF5",
    "factory": "#2EC4B6",
    "maintenance": "#F4A261",
    "procurement": "#E76F51",
}
PALETTE = ["#4C8BF5", "#2EC4B6", "#F4A261", "#E76F51", "#9B5DE5"]
SILO_KEY = {
    "Sales CRM": "sales",
    "Factory Floor": "factory",
    "Maintenance": "maintenance",
    "Procurement": "procurement",
}

Parse JSON from an LLM reply.

In [ ]:
def parse_json_blob(text: str) -> dict:
    """Pull a JSON object from an LLM reply (tolerates fences / trailing commas)."""
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*", "", t)
    t = re.sub(r"\s*```$", "", t)
    start, end = t.find("{"), t.rfind("}")
    if start < 0 or end < 0:
        raise ValueError(f"no JSON object in reply:\n{text[:400]}")
    blob = t[start : end + 1]
    blob = re.sub(r",\s*([}\]])", r"\1", blob)  # trailing commas
    try:
        return json.loads(blob)
    except json.JSONDecodeError as e:
        raise json.JSONDecodeError(
            f"{e.msg} — raw was:\n{blob[:500]}",
            e.doc,
            e.pos,
        ) from None


Extraction system prompt.

In [ ]:
EXTRACT_SYSTEM = """
Extract entities and relations from one short document.

Return ONLY a JSON object with this exact schema (no markdown, no extra keys, no comments):
{
  "entities": [{"id": "snake_or_Pascal", "label": "short name", "type": "thing"}],
  "relations": [{"source": "id", "target": "id", "relation": "verb_snake"}]
}

Rules:
- Use names as written in the document (e.g. if it says Press Cell 4, id can be Press_Cell_4).
- Keep ids consistent with wording so the same real-world thing gets the same id string.
- At most 3 entities and 2 relations.
- No dates as entities.
- Double-quoted JSON only; no trailing commas.
""".strip()


Extract from one chunk via LLM.

In [ ]:
def extract_from_chunk(chunk: dict) -> dict:
    prompt = (
        f"Silo: {chunk['silo']}\nTitle: {chunk['title']}\n"
        f"Document: {chunk['text']}\n\n"
        "Return the JSON object now."
    )
    raw = ask_llm(prompt, system=EXTRACT_SYSTEM, json_mode=True)
    try:
        data = parse_json_blob(raw)
    except (json.JSONDecodeError, ValueError):
        # one repair pass
        raw = ask_llm(
            "Fix this into valid JSON matching the schema only:\n" + raw,
            system=EXTRACT_SYSTEM,
            json_mode=True,
        )
        data = parse_json_blob(raw)
    if "entities" not in data or "relations" not in data:
        raise ValueError(f"missing keys in extraction: {data.keys()}")
    data["source"] = PROVIDER
    return data


Run extraction on every chunk.

In [ ]:
extractions = {}
for ch in CHUNKS:
    data = extract_from_chunk(ch)
    extractions[ch["id"]] = data
    ents = [e["id"] for e in data.get("entities", [])]
    rels = [f"{r['source']}-[{r['relation']}]->{r['target']}" for r in data.get("relations", [])]
    print(f"{ch['id']}: {ents} | {rels}")


Merge fragments on shared entity ids.

In [ ]:
def merge_extractions(chunks, extractions) -> dict:
    nodes, edges, seen_e, seen_r = [], [], set(), set()
    for ch in chunks:
        silo = SILO_KEY.get(ch["silo"], "sales")
        data = extractions[ch["id"]]
        for e in data.get("entities", []):
            eid = e["id"]
            if eid in seen_e:
                continue
            seen_e.add(eid)
            nodes.append(
                {
                    "id": eid,
                    "label": e.get("label", eid),
                    "type": e.get("type", "entity"),
                    "silo": silo,
                    "from_chunk": ch["id"],
                }
            )
        for r in data.get("relations", []):
            key = (r["source"], r["target"], r.get("relation", "related"))
            if key in seen_r:
                continue
            if r["source"] not in seen_e or r["target"] not in seen_e:
                # still keep edge if endpoints appear under same string later — add stubs
                pass
            seen_r.add(key)
            edges.append(
                {
                    "source": r["source"],
                    "target": r["target"],
                    "relation": r.get("relation", "related"),
                    "from_chunk": ch["id"],
                }
            )
    # ensure edge endpoints exist as nodes
    have = {n["id"] for n in nodes}
    for e in edges:
        for endpoint in (e["source"], e["target"]):
            if endpoint not in have:
                have.add(endpoint)
                nodes.append(
                    {
                        "id": endpoint,
                        "label": endpoint,
                        "type": "entity",
                        "silo": "shared",
                        "from_chunk": e["from_chunk"],
                    }
                )
    seed = next((n["id"] for n in nodes if "402" in n["id"] or "Order" in n["id"]), nodes[0]["id"] if nodes else None)
    return {"nodes": nodes, "edges": edges, "default_seed": seed}


Build `BMW` graph dict.

In [ ]:
BMW = merge_extractions(CHUNKS, extractions)
print(f"{len(BMW['nodes'])} nodes, {len(BMW['edges'])} edges")
display(pd.DataFrame(BMW["nodes"])[["id", "label", "silo", "from_chunk"]])
pd.DataFrame(BMW["edges"])


NetworkX builder.

In [ ]:
def build_graph(graph_doc: dict, directed: bool = True):
    g = nx.DiGraph() if directed else nx.Graph()
    for n in graph_doc["nodes"]:
        g.add_node(n["id"], **{k: v for k, v in n.items() if k != "id"})
    for e in graph_doc["edges"]:
        g.add_edge(e["source"], e["target"], relation=e.get("relation", "related"))
    return g

In [ ]:
def draw_graph(graph_doc, assignment=None, title="", height="480px"):
    """Draw KG in-notebook with matplotlib (reliable in Cursor).

    Also writes an interactive pyvis HTML under ._grag_viz/ — open that in a
    browser if you want drag/zoom. (pyvis iframes often render blank in Cursor.)
    """
    g = build_graph(graph_doc, directed=True)
    pos = nx.spring_layout(g, seed=42)

    if assignment is not None:
        colors = [
            PALETTE[int(assignment[n]) % len(PALETTE)] if n in assignment else "#888888"
            for n in g.nodes()
        ]
    else:
        colors = ["#4C8BF5"] * g.number_of_nodes()

    labels = {n: str(g.nodes[n].get("label") or n) for n in g.nodes()}
    edge_labels = {(u, v): d.get("relation", "") for u, v, d in g.edges(data=True)}

    fig, ax = plt.subplots(figsize=(9, 6))
    nx.draw_networkx_edges(g, pos, ax=ax, alpha=0.45, arrows=True, arrowsize=14)
    nx.draw_networkx_nodes(g, pos, ax=ax, node_color=colors, node_size=900, edgecolors="white")
    nx.draw_networkx_labels(g, pos, labels=labels, ax=ax, font_size=8)
    if g.number_of_edges() <= 40:
        nx.draw_networkx_edge_labels(g, pos, edge_labels=edge_labels, ax=ax, font_size=7)
    if title:
        ax.set_title(title)
    ax.set_axis_off()
    fig.tight_layout()
    display(fig)
    plt.close(fig)

    # Interactive HTML for browser (optional)
    try:
        net = Network(
            height=height,
            width="100%",
            bgcolor="#ffffff",
            font_color="#222",
            directed=True,
            notebook=False,
            cdn_resources="in_line",
        )
        net.barnes_hut(gravity=-8000, central_gravity=0.2, spring_length=120, spring_strength=0.02, damping=0.4)
        for n, data in g.nodes(data=True):
            color = (
                PALETTE[int(assignment[n]) % len(PALETTE)]
                if assignment is not None and n in assignment
                else "#4C8BF5"
            )
            net.add_node(n, label=labels[n], title=labels[n], color=color, size=22)
        for u, v, data in g.edges(data=True):
            rel = data.get("relation", "")
            net.add_edge(u, v, title=rel, label=rel if g.number_of_edges() <= 40 else "")
        out_dir = Path("._grag_viz")
        out_dir.mkdir(exist_ok=True)
        safe = "".join(ch if ch.isalnum() or ch in "-_" else "_" for ch in (title or "graph"))[:40]
        out = out_dir / f"{safe or 'graph'}.html"
        out.write_text(net.generate_html(), encoding="utf-8")
        print(f"Interactive (browser): {out.resolve()}")
    except Exception as e:
        print("pyvis HTML skip:", type(e).__name__, e)


Print edges (with relation text).


In [ ]:
g = build_graph(BMW)
for u, v, d in g.edges(data=True):
    print(f"{u} --[{d['relation']}]--> {v}")


Draw the merged KG.

In [ ]:
draw_graph(BMW, title="Extracted KG (ids, one colour)")


## 6. Clustering

Flat Louvain on a denser BMW-story graph (mock data — extracted `BMW` is too small).  
Next section stacks **another** Louvain on top → hierarchy (2 levels).


Mock community graph (same delay story, more nodes).


In [ ]:
COMM_GRAPH = {
    "nodes": [
        {"id": "Order_402", "silo": "sales"},
        {"id": "Sales_Escalation", "silo": "sales"},
        {"id": "Penalty_Waiver", "silo": "sales"},
        {"id": "CRM_Ticket", "silo": "sales"},
        {"id": "Press_Cell_4", "silo": "factory"},
        {"id": "Chassis_Line", "silo": "factory"},
        {"id": "Shift_Aug2", "silo": "factory"},
        {"id": "Die_Set_B", "silo": "factory"},
        {"id": "Hydraulic_Pump", "silo": "maintenance"},
        {"id": "Pump_Seal", "silo": "maintenance"},
        {"id": "MTTR_Ticket", "silo": "maintenance"},
        {"id": "Spare_Kit", "silo": "maintenance"},
        {"id": "Customs_Hold", "silo": "procurement"},
        {"id": "Port_Broker", "silo": "procurement"},
        {"id": "Seal_PO", "silo": "procurement"},
        {"id": "Air_Freight", "silo": "procurement"},
    ],
    "edges": [
        # sales cluster
        {"source": "Order_402", "target": "Sales_Escalation", "relation": "reported_in"},
        {"source": "Sales_Escalation", "target": "Penalty_Waiver", "relation": "requests"},
        {"source": "Sales_Escalation", "target": "CRM_Ticket", "relation": "logged_as"},
        {"source": "Order_402", "target": "CRM_Ticket", "relation": "tracked_in"},
        # factory cluster
        {"source": "Press_Cell_4", "target": "Chassis_Line", "relation": "on_line"},
        {"source": "Chassis_Line", "target": "Shift_Aug2", "relation": "ran_on"},
        {"source": "Press_Cell_4", "target": "Die_Set_B", "relation": "uses"},
        {"source": "Die_Set_B", "target": "Shift_Aug2", "relation": "setup_for"},
        # maintenance cluster
        {"source": "Hydraulic_Pump", "target": "Pump_Seal", "relation": "needs"},
        {"source": "Pump_Seal", "target": "MTTR_Ticket", "relation": "logged_as"},
        {"source": "MTTR_Ticket", "target": "Spare_Kit", "relation": "requests"},
        {"source": "Hydraulic_Pump", "target": "Spare_Kit", "relation": "uses"},
        # procurement cluster
        {"source": "Customs_Hold", "target": "Port_Broker", "relation": "handled_by"},
        {"source": "Port_Broker", "target": "Seal_PO", "relation": "against"},
        {"source": "Seal_PO", "target": "Air_Freight", "relation": "expedite"},
        {"source": "Customs_Hold", "target": "Seal_PO", "relation": "blocks"},
        # bridges (causal chain across silos)
        {"source": "Order_402", "target": "Press_Cell_4", "relation": "routed_to"},
        {"source": "Press_Cell_4", "target": "Hydraulic_Pump", "relation": "halted_by"},
        {"source": "Pump_Seal", "target": "Customs_Hold", "relation": "blocked_by"},
    ],
    "default_seed": "Order_402",
}
print(len(COMM_GRAPH["nodes"]), "nodes", len(COMM_GRAPH["edges"]), "edges")


Draw it once — all one colour (before clustering).


In [ ]:
draw_graph(COMM_GRAPH, title="Before clustering")


Louvain community detection.


In [ ]:
def cluster_graph(graph_doc: dict) -> dict:
    g = build_graph(graph_doc, directed=False)
    communities = list(nx_comm.louvain_communities(g, seed=42))
    communities = [sorted(c) for c in communities]
    communities.sort(key=lambda c: (-len(c), c[0] if c else ""))
    assignment = {n: i for i, members in enumerate(communities) for n in members}
    return {
        "method": "louvain",
        "n_communities": len(communities),
        "communities": communities,
        "assignment": assignment,
    }


Run clustering — raw result.


In [ ]:
clustered = cluster_graph(COMM_GRAPH)
clustered


Same graph, coloured by community.


In [ ]:
draw_graph(COMM_GRAPH, assignment=clustered["assignment"], title="Communities")


## 7. Hierarchical communities


| Level | What | Meaning |
|-------|------|---------|
| **0 (fine)** | Louvain on entity graph | tight local groups |
| **1 (coarse)** | contract communities → Louvain again | broader themes |

Same `COMM_GRAPH` / `clustered` from §6 = Level 0.



### Level 0 — fine communities (already computed)


In [ ]:
level0 = clustered
print("level", 0)
print("n_communities", level0["n_communities"])
print("communities", level0["communities"])


In [ ]:
draw_graph(COMM_GRAPH, assignment=level0["assignment"], title="Level 0 (fine)")


### Contract — each Level-0 community becomes one supernode


In [ ]:
def contract_communities(graph_doc, clustered):
    nodes = [{"id": f"c{i}", "label": f"C{i}"} for i in range(clustered["n_communities"])]
    edges, seen = [], set()
    g = build_graph(graph_doc, directed=False)
    assign = clustered["assignment"]
    for u, v in g.edges():
        a, b = sorted((assign[u], assign[v]))
        if a == b or (a, b) in seen:
            continue
        seen.add((a, b))
        edges.append({"source": f"c{a}", "target": f"c{b}", "relation": "adjacent"})
    return {"nodes": nodes, "edges": edges}


In [ ]:
coarse = contract_communities(COMM_GRAPH, level0)
print(coarse)


In [ ]:
draw_graph(coarse, title="After contract (supernodes)")


### Level 1 — Louvain again on the supernode graph


In [ ]:
level1 = cluster_graph(coarse)
print("level", 1)
print("n_communities", level1["n_communities"])
print("communities", level1["communities"])
print("assignment", level1["assignment"])


In [ ]:
draw_graph(coarse, assignment=level1["assignment"], title="Level 1 (coarse)")


Same Level-1 colours on the **full entity graph** (each node inherits its supernode's Level-1 community).


In [ ]:
# map entity -> level0 community id -> level1 community id
level1_on_entities = {}
for node, l0 in level0["assignment"].items():
    supernode = f"c{l0}"
    level1_on_entities[node] = level1["assignment"][supernode]

print(level1_on_entities)
draw_graph(COMM_GRAPH, assignment=level1_on_entities, title="Level 1 on entity graph")


### Two levels side by side


In [ ]:
print("LEVEL 0 (fine entities):")
for i, members in enumerate(level0["communities"]):
    print(i, members)

print("\nLEVEL 1 (groups of Level-0 communities):")
for i, members in enumerate(level1["communities"]):
    print(i, members)


## 8. Community summaries

LLM one-liners for each Louvain community (used later for global search).


Summarize helper.


In [ ]:
def summarize_one(cid, member_ids):
    prompt = f"Community members: {member_ids}. One sentence on this BMW delay theme."
    text = ask_llm(prompt, system="One terse community report sentence.")
    return {"id": f"C{cid}", "members": member_ids, "summary": text}


Build reports for each level-0 community.


In [ ]:
reports = []
for cid, members in enumerate(clustered["communities"]):
    reports.append(summarize_one(cid, members))
reports


## 9. Retrieval

**Local GraphRAG:**
1. Embed query ↔ nodes → **seed**
2. **k-hop** neighbourhood
3. Load **source chunks** linked to those nodes (+ edges) → LLM

**Global:** embed-rank **community reports** (themes).

Local uses `COMM_GRAPH` so hops are visible.


### Local graph


In [ ]:
draw_graph(COMM_GRAPH, title="Local-search graph")


Local question (entity-flavoured).


In [ ]:
local_q = "What halted Press Cell 4?"
local_q


Embed the query.


In [ ]:
q_local = embed([local_q])[0]
len(q_local)


Embed every node id (stand-in for entity text).


In [ ]:
node_ids = [n["id"] for n in COMM_GRAPH["nodes"]]
node_vecs = embed(node_ids)
list(zip(node_ids, [len(v) for v in node_vecs]))


Cosine(query, each node) → pick seed = best match.


In [ ]:
node_scores = sorted(
    [
        {"id": nid, "score": round(cosine(q_local, nv), 4)}
        for nid, nv in zip(node_ids, node_vecs)
    ],
    key=lambda r: -r["score"],
)
node_scores


In [ ]:
seed = node_scores[0]["id"]
seed


k-hop from the **embedding-chosen** seed.


In [ ]:
def local_khop(seed, k, graph_doc):
    g = build_graph(graph_doc, directed=False)
    distances = nx.single_source_shortest_path_length(g, seed, cutoff=k)
    return {"seed": seed, "k": k, "distances": dict(sorted(distances.items(), key=lambda kv: kv[1]))}


**k=1** — immediate neighbours only.


In [ ]:
local_k1 = local_khop(seed, 1, COMM_GRAPH)
local_k1


**k=3** — deeper neighbourhood (still not a global theme).


In [ ]:
local_k3 = local_khop(seed, 3, COMM_GRAPH)
local_k3


Plot k=3 neighbourhood (seed vs reached).


In [ ]:
reached = local_k3["distances"]
local_assign = {n: (0 if d == 0 else 1) for n, d in reached.items()}
draw_graph(COMM_GRAPH, assignment=local_assign, title=f"Local k=3 from seed={seed}")


### Pull source chunks for neighbourhood nodes

Each entity points back to document text and those **chunks** are what the LLM reads (plus structure).


Map node → silo → chunk (in production: entity→chunk links from extraction).


In [ ]:
SILO_TO_CHUNK = {"sales": "A", "factory": "B", "maintenance": "C", "procurement": "D"}
chunk_by_id = {c["id"]: c for c in CHUNKS}
node_silo = {n["id"]: n.get("silo") for n in COMM_GRAPH["nodes"]}
node_silo


Chunks linked to the **k=1** neighbourhood (seed + neighbours).


In [ ]:
nbr_nodes = list(local_k1["distances"].keys())
chunk_ids = sorted({
    SILO_TO_CHUNK[node_silo[n]]
    for n in nbr_nodes
    if node_silo.get(n) in SILO_TO_CHUNK
})
local_chunks = [chunk_by_id[i] for i in chunk_ids]
chunk_ids, local_chunks


Edges in that neighbourhood (structure).


In [ ]:
local_edges = [
    e for e in COMM_GRAPH["edges"]
    if e["source"] in local_k1["distances"] and e["target"] in local_k1["distances"]
]
local_edges


LLM context = chunks + edges (not ids alone).


In [ ]:
local_prompt = (
    f"Question: {local_q}\n"
    f"Seed (via embed): {seed}\n"
    f"Neighbourhood nodes: {nbr_nodes}\n"
    f"Edges: {local_edges}\n"
    "Chunks:\n"
    + "\n".join(f"[{c['id']}] {c['text']}" for c in local_chunks)
    + "\nAnswer in 2 sentences using only these chunks and edges."
)
local_prompt


In [ ]:
local_answer = ask_llm(local_prompt, system="Use only the provided chunks and edges.")
local_answer


### Global — theme question over community reports


In [ ]:
theme_q = "What themes drove production delays and customs issues across the plant?"
theme_q


In [ ]:
report_texts = [r["summary"] + " " + " ".join(r["members"]) for r in reports]
report_texts


In [ ]:
qv = embed([theme_q])[0]
rvs = embed(report_texts)
scores = [round(cosine(qv, rv), 4) for rv in rvs]
scores


In [ ]:
ranked_reports = sorted(
    [{**r, "score": s} for r, s in zip(reports, scores)],
    key=lambda r: -r["score"],
)
ranked_reports


In [ ]:
top_reports = ranked_reports[:2]
top_reports


In [ ]:
global_prompt = (
    f"Question: {theme_q}\n\n"
    + "\n".join(f"- {r['id']}: {r['summary']}" for r in top_reports)
    + "\n\nAnswer in 2-3 sentences using only these reports."
)
global_prompt


In [ ]:
global_answer = ask_llm(global_prompt, system="Use only the community reports.")
global_answer


### Compare — local neighbourhood vs global themes vs vector chunks


In [ ]:
local_q


In [ ]:
local_answer


In [ ]:
theme_q


In [ ]:
global_answer


In [ ]:
QUERY


In [ ]:
vector_rag_answer


## 10. Cost — asymptotic LLM-call comparison

Let  
- $C$ = number of chunks  
- $N$ = number of graph nodes (entities)  
- $K$ = number of communities (sum over hierarchy levels in production)  
- $Q$ = number of queries  

Unit = **LLM chat calls** (embeddings omitted; local/free or a cheaper API).

| Stage | Vector RAG | GraphRAG (this lab) | GraphRAG (typical production) |
|--|--|--|--|
| **Index** | $O(1)$ chat *(embed only)* | $O(C + K_{L0})$ — extract per chunk + summarize L0 | $O(C + K)$ — extract (+ repair) + summarize **all** levels; often + entity-resolve / claims |
| **Per query** | $O(1)$ | $O(1)$ local or global | $O(1)$ local; global map-reduce can be $O(K')$ over ranked communities |
| **Total after $Q$ queries** | $O(Q)$ | $O(C + K_{L0} + Q)$ | $O(C + K + Q)$ *(or $O(C + K + Q \cdot K')$ if map-reduce global)* |

 GraphRAG pays **$O(C+K)$ up front**; each query stays **$O(1)$** if reports are cached.


## 11. Take Home Challenge

- Implement DRIFT combining both local and global modes.

You can refer :
https://microsoft.github.io/graphrag/query/drift_search/